# z719 final gerencial v2: hiperparámetros elegidos por ganancia
Francisco Medina Raed. Sigue al WF7190 (V3 desde 202103 + semillerío de 20), con un cambio central: **el grid search elige por ganancia en 202107 y no por AUC**.

**Por qué:** en el Exp 07 y en el WF7190 el AUC de validación fue al revés de la ganancia. En el WF7190, V2 eligió `min_data_in_leaf = 32` por tener el mejor AUC y la ganancia Public se desplomó.

**Qué cambia respecto del WF7190**

| | WF7190 | WF7191 (este) |
|---|---|---|
| Criterio del grid search | AUC en 202107 | **Ganancia en 202107**: 975.000 por BAJA+2 contactado, −25.000 por BAJA+1 o CONTINUA contactado |
| Early stopping | Por AUC | **Por ganancia** (`feval`) |
| Evaluación de cada combinación | 1 semilla | **3 semillas**, promediadas (la ganancia de un mes es ruidosa) |
| Grilla | 5 × 7, con `min_data_in_leaf` 16 y 32 | 5 × 6, `min_data_in_leaf` de 64 a 2048 |
| Variantes | V3 y V2 | Solo V3 (desde 202103) |
| Corte de referencia | — | El grid informa el **corte óptimo en julio** |
| Curva de puntajes | Solo el WF7190 | WF7191 **junto al WF7190**, para comparar |

**La ganancia en validación:** se ordena a los clientes de 202107 por probabilidad, se acumula la ganancia de contactar a los primeros N y se toma el máximo de esa curva **suavizada con un promedio móvil de 401 clientes**. Así no se elige por un pico aislado.

Todo lo demás es igual al WF7190: preprocesamiento, semillerío de 20 semillas, cortes de 800 a 1.400 de a 50, reanudable.

**Tiempo:** 30 combinaciones × 3 semillas = 90 entrenamientos en el grid, unas 2,5 veces el WF7190. Primero conviene correr `modo <- "test"`.

Opción a probar en otra corrida: `PARAM$final$escalar_iteraciones <- TRUE` escala `num_iterations` por las filas que suma 202107 al modelo final. Está en `FALSE` para cambiar una sola cosa por vez.

## Parámetros, paquetes y carpetas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

modo <- "full"           # "test" = corrida corta para probar que todo anda (no sube a Kaggle)
subir_kaggle <- TRUE     # sube los archivos a Kaggle apenas se generan (necesita el kaggle.json)

PARAM <- list()
PARAM$semilla_primigenia <- 123457     # <- tu semilla primigenia
PARAM$experimento <- 7191
PARAM$comparar_con <- 7190            # experimentos anteriores que se muestran en la curva de puntajes
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Training strategy: la ganadora del Exp 07 y del WF7190 ---
PARAM$variantes <- list(
  list(nombre = "V3gan_desde202103", desde = 202103,
       descripcion = "training 202103-202106, final_train 202103-202107, hiperparametros elegidos por ganancia en 202107")
)
PARAM$trainingstrategy$validate     <- 202107
PARAM$trainingstrategy$hasta        <- 202106   # ultimo mes de training
PARAM$trainingstrategy$future       <- 202109
PARAM$trainingstrategy$training_pct <- 1.0      # sin undersampling, como el z719
PARAM$trainingstrategy$positivos    <- c("BAJA+1", "BAJA+2")

# --- Ganancia de la competencia ---
PARAM$ganancia$acierto <- 975000    # por cada BAJA+2 contactado
PARAM$ganancia$error   <- -25000    # por cada BAJA+1 o CONTINUA contactado
PARAM$ganancia$ventana <- 401       # suavizado de la curva de ganancia en validacion (clientes)

# --- Grid search: ahora se elige por GANANCIA en 202107, no por AUC ---
# min_data_in_leaf arranca en 64: con 16 y 32 el WF7190 sobreajusto (V2 se desplomo en Kaggle)
PARAM$grid <- list(
  num_leaves       = c(64, 128, 256, 512, 1024),
  min_data_in_leaf = c(64, 128, 256, 512, 1024, 2048)
)
PARAM$grid$semillas_eval <- 3       # cada combinacion se evalua con 3 semillas y se promedia (la ganancia es ruidosa)

# --- Parametros fijos de LightGBM: identicos al z719 ---
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "None",              # la metrica es la ganancia (feval), no el AUC
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 31,
  learning_rate = 0.03,
  feature_fraction = 0.5,
  num_iterations = 2048,        # lo limita el early stopping
  early_stopping_rounds = 200,
  num_leaves = 64,
  min_data_in_leaf = 128
)

# --- Semillerio: el modelo final se entrena con N semillas y se promedian las probabilidades ---
PARAM$semillerio$cantidad <- 20

# --- Opcional: escalar num_iterations por las filas que agrega 202107 al final_train ---
PARAM$final$escalar_iteraciones <- FALSE

# --- Kaggle: cortes de a 50 alrededor del pico del Exp 07 (unos 1.100 envios) ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)

if (modo == "test") {
  PARAM$experimento <- paste0(PARAM$experimento, "_test")
  PARAM$grid <- list(num_leaves = c(64, 256), min_data_in_leaf = c(64, 512), semillas_eval = 2)
  PARAM$lgbm$param_fijos$num_iterations <- 60
  PARAM$lgbm$param_fijos$early_stopping_rounds <- 20
  PARAM$semillerio$cantidad <- 3
  subir_kaggle <- FALSE
}

# semillas del semillerio: la primera es la primigenia, el resto se derivan de ella (reproducible)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

# carpeta del experimento en el bucket de la VM (o en el home si no hay bucket)
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta_exp <- file.path(base, paste0("WF", PARAM$experimento))
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)

# dataset: lo busco donde lo suele dejar la VM; si no esta, lo descargo
candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta_exp), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta_exp, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}

message("carpeta: ", carpeta_exp, "\ndataset: ", archivo_dataset,
        "\nsemillas del semillerio: ", paste(PARAM$semillerio$semillas, collapse = ", "))

## 6.3.1 Preprocesamiento
Idéntico al z719 y al WF7190.

In [ ]:
dataset <- fread(archivo_dataset)

# 6.3.1.2 Catastrophe Analysis (identico al z719)
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# 6.3.1.3 FE intra-mes (identico al z719)
atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  length(atributos) == length(intersect(atributos, colnames(dataset)))
}
if (atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]
if (atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# 6.3.1.5 FE historico: lags y deltas de orden 1 y 2 (identico al z719)
# los lags usan la fila anterior de cada cliente: verifico que esten en orden cronologico
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# clase binaria y azar para el (eventual) undersampling, como el z719
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1L, 0L)]
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]

campos_buenos <- copy(setdiff(colnames(dataset),
  c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")))

message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ",
        nrow(dataset), " filas x ", length(campos_buenos), " variables")

## Funciones
- **`ganancia_maxima`**: ganancia máxima de la curva suavizada y el corte donde ocurre.
- **`grid_search`**: para cada combinación entrena con 3 semillas, con early stopping por ganancia en 202107, y guarda la ganancia promedio, su desvío, el corte y las iteraciones.
- **`semillerio`** y **`generar_kaggle`**: iguales al WF7190 (con la opción de escalar iteraciones).

In [ ]:
# ---------- Kaggle ----------
preparar_kaggle <- function() {
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "")
    invisible(system("python3 -m pip install -q --user kaggle"))
}

# sube un archivo; si falla (cupo diario, red) lo anota en pendientes_kaggle.txt para reintentar.
# los que ya subieron quedan en subidos_kaggle.txt y no se vuelven a subir al re-correr
# si una subida falla, el resto de la corrida ya no intenta: todo queda pendiente para la proxima vez
kaggle_sin_cupo <- FALSE
subir_a_kaggle <- function(archivo, mensaje) {
  archivo <- normalizePath(archivo)
  if (file.exists(file.path(carpeta_exp, "subidos_kaggle.txt")) &&
      archivo %in% readLines(file.path(carpeta_exp, "subidos_kaggle.txt"))) return(TRUE)
  if (kaggle_sin_cupo) {
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
    return(FALSE)
  }
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                         "-f", shQuote(archivo), "-m", shQuote(mensaje)),
                             stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) {
    cat(archivo, "\n", sep = "", file = file.path(carpeta_exp, "subidos_kaggle.txt"), append = TRUE)
    Sys.sleep(30)
  } else {
    kaggle_sin_cupo <<- TRUE
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
  }
  ok
}

reintentar_pendientes <- function() {
  arch <- file.path(carpeta_exp, "pendientes_kaggle.txt")
  if (!subir_kaggle || !file.exists(arch)) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines(arch)), ""))
  file.remove(arch)
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (p in pendientes) {          # si falla uno, los siguientes quedan anotados sin intentar
    partes <- strsplit(p, "|", fixed = TRUE)[[1]]
    subir_a_kaggle(partes[1], partes[2])
  }
}

# ---------- Training strategy ----------
meses_training <- function(desde) {
  meses <- sort(unique(dataset$foto_mes), decreasing = TRUE)
  meses[meses >= desde & meses <= PARAM$trainingstrategy$hasta]
}

lgb_dataset <- function(filtro) {
  lgb.Dataset(data = data.matrix(dataset[filtro, campos_buenos, with = FALSE]),
              label = dataset[filtro, clase01], free_raw_data = TRUE)
}

# ---------- Ganancia ----------
# ordena por probabilidad, acumula la ganancia de contactar a los primeros N y devuelve el maximo
# de la curva suavizada (promedio movil de PARAM$ganancia$ventana clientes) y el corte donde ocurre
ganancia_maxima <- function(prob, gan) {
  orden <- order(prob, decreasing = TRUE)
  curva <- cumsum(gan[orden])
  suave <- frollmean(curva, PARAM$ganancia$ventana, align = "center")
  corte <- which.max(suave)
  list(ganancia = suave[corte], corte = corte)
}

# ---------- 6.3.2.2 Grid search por GANANCIA en validacion (reanudable) ----------
grid_search <- function(variante, carpeta) {
  archivo <- file.path(carpeta, "tb_grid_search.txt")
  tb <- CJ(num_leaves = PARAM$grid$num_leaves, min_data_in_leaf = PARAM$grid$min_data_in_leaf)
  if (file.exists(archivo)) tb <- tb[!fread(archivo), on = .(num_leaves, min_data_in_leaf)]

  if (nrow(tb)) {
    training <- meses_training(variante$desde)
    filtro_train <- dataset[, foto_mes %in% training &
      (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]
    filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
    dtrain <- lgb_dataset(filtro_train)
    dvalidate <- lgb_dataset(filtro_valid)
    mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE])
    # ganancia de contactar a cada cliente de validacion, en el mismo orden que dvalidate
    gan_valid <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]

    # metrica para el early stopping: la ganancia maxima suavizada en validacion
    feval_ganancia <- function(preds, dtrain) {
      list(name = "ganancia", value = ganancia_maxima(preds, gan_valid)$ganancia, higher_better = TRUE)
    }

    semillas_eval <- PARAM$semillerio$semillas[seq_len(PARAM$grid$semillas_eval)]
    for (i in seq_len(nrow(tb))) {
      x <- as.list(tb[i])
      evals <- rbindlist(lapply(semillas_eval, function(s) {
        param <- modifyList(PARAM$lgbm$param_fijos, c(x, list(seed = s)))
        modelo <- lgb.train(data = dtrain, valids = list(valid = dvalidate), eval = feval_ganancia,
                            param = param, verbose = -100)
        niter <- modelo$best_iter
        g <- ganancia_maxima(predict(modelo, mvalid, num_iteration = niter), gan_valid)
        rm(modelo); gc(full = TRUE, verbose = FALSE)
        data.table(semilla = s, ganancia = g$ganancia, corte = g$corte, num_iterations = niter)
      }))
      fila <- data.table(tb[i], ganancia = mean(evals$ganancia), ganancia_sd = sd(evals$ganancia),
                         corte = round(mean(evals$corte)), num_iterations = round(mean(evals$num_iterations)))
      fwrite(fila, archivo, sep = "\t", append = file.exists(archivo))
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  nl=", x$num_leaves,
              " mdil=", x$min_data_in_leaf, "  niter ", fila$num_iterations, "  corte ", fila$corte,
              "  ganancia ", round(fila$ganancia / 1e6, 2), " M")
    }
    rm(dtrain, dvalidate, mvalid); gc(full = TRUE, verbose = FALSE)
  }
  res <- fread(archivo)
  setorder(res, -ganancia)
  res
}

# ---------- 6.3.3 Final training con semillerio (reanudable: una prediccion por semilla) ----------
semillerio <- function(variante, carpeta, mejores) {
  carpeta_sem <- file.path(carpeta, "semillas")
  dir.create(carpeta_sem, showWarnings = FALSE)

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  # modifyList: los mejores del grid PISAN a los fijos (con c() quedaban duplicados y LightGBM usaba los fijos)
  param_final <- modifyList(fijos, mejores)

  pendientes <- PARAM$semillerio$semillas[
    !file.exists(file.path(carpeta_sem, paste0("pred_s", PARAM$semillerio$semillas, ".txt")))]

  final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  if (isTRUE(PARAM$final$escalar_iteraciones)) {
    # el grid eligio las iteraciones con los meses de training; el final_train suma 202107
    factor <- dataset[foto_mes %in% final_train, .N] / dataset[foto_mes %in% meses_training(variante$desde), .N]
    param_final$num_iterations <- round(param_final$num_iterations * factor)
  }

  if (length(pendientes)) {
    dfinal_train <- lgb_dataset(dataset[, foto_mes %in% final_train])
    dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
    mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (s in pendientes) {
      param_final$seed <- s
      modelo <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
      fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
             file.path(carpeta_sem, paste0("pred_s", s, ".txt")), sep = "\t")
      if (s == PARAM$semilla_primigenia) {          # modelo e importancia solo de la semilla primigenia
        lgb.save(modelo, file.path(carpeta, "modelo.txt"))
        fwrite(as.data.table(lgb.importance(modelo)), file.path(carpeta, "impo.txt"), sep = "\t")
      }
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  semilla ", s, " lista")
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dfinal_train, dfuture, mfuture); gc(full = TRUE, verbose = FALSE)
  }

  # promedio de las probabilidades de todas las semillas
  preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s)
    fread(file.path(carpeta_sem, paste0("pred_s", s, ".txt")))[, semilla := s]))
  tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
  fwrite(tb_prediccion, file.path(carpeta, "prediccion.txt"), sep = "\t")   # util para ensembles
  list(prediccion = tb_prediccion, param_final = param_final)
}

# ---------- Archivos para Kaggle ----------
generar_kaggle <- function(variante, carpeta, tb_prediccion) {
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  tb <- copy(tb_prediccion)
  setorder(tb, -prob)
  for (envios in PARAM$kaggle$cortes) {
    tb[, Predicted := 0L]
    tb[1:envios, Predicted := 1L]
    archivo <- file.path(carpeta, "kaggle",
                         paste0("KA", PARAM$experimento, "_", variante$nombre, "_", envios, ".csv"))
    fwrite(tb[, .(numero_de_cliente, Predicted)], archivo, sep = ",")
    if (subir_kaggle)
      subir_a_kaggle(archivo, paste0("z719 ", variante$nombre, " semillerio=", PARAM$semillerio$cantidad,
                                     " semilla=", PARAM$semilla_primigenia, " envios=", envios))
  }
}

## Corridas

In [ ]:
if (subir_kaggle) { preparar_kaggle(); reintentar_pendientes() }
t0 <- Sys.time()
resumen <- list()

for (variante in PARAM$variantes) {
  carpeta <- file.path(carpeta_exp, variante$nombre)
  dir.create(carpeta, showWarnings = FALSE)
  message(format(Sys.time(), "%a %b %d %X %Y"), "  ===== ", variante$nombre,
          ": training desde ", variante$desde, " =====")

  # grid search
  tb_grid <- grid_search(variante, carpeta)
  mejores <- as.list(tb_grid[1, .(num_leaves, min_data_in_leaf, num_iterations)])
  message("  mejores: nl=", mejores$num_leaves, " mdil=", mejores$min_data_in_leaf,
          " niter=", mejores$num_iterations, "  ganancia en ", PARAM$trainingstrategy$validate, ": ",
          round(tb_grid[1, ganancia] / 1e6, 2), " M (sd entre semillas ", round(tb_grid[1, ganancia_sd] / 1e6, 2),
          " M) en ", tb_grid[1, corte], " envios")
  print(head(tb_grid, 5))
  if (mejores$num_leaves %in% range(PARAM$grid$num_leaves) ||
      mejores$min_data_in_leaf %in% range(PARAM$grid$min_data_in_leaf))
    warning(variante$nombre, ": el optimo cayo en el borde de la grilla")

  # final training con semillerio y prediccion de 202109
  fin <- semillerio(variante, carpeta, mejores)

  # archivos y subida a Kaggle
  generar_kaggle(variante, carpeta, fin$prediccion)

  # registro de la corrida
  PARAM_var <- PARAM
  PARAM_var$variante <- variante
  PARAM_var$trainingstrategy$training <- meses_training(variante$desde)
  PARAM_var$trainingstrategy$final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  PARAM_var$out$lgbm$ganancia_validacion <- tb_grid[1, ganancia]
  PARAM_var$out$lgbm$corte_validacion <- tb_grid[1, corte]
  PARAM_var$out$lgbm$mejores_hiperparametros <- mejores
  PARAM_var$out$param_final <- fin$param_final
  write_yaml(PARAM_var, file.path(carpeta, "PARAM.yml"))

  resumen[[variante$nombre]] <- data.table(variante = variante$nombre, desde = variante$desde,
    ganancia_valid_M = round(tb_grid[1, ganancia] / 1e6, 2), corte_valid = tb_grid[1, corte], num_leaves = mejores$num_leaves,
    min_data_in_leaf = mejores$min_data_in_leaf, num_iterations = mejores$num_iterations,
    num_iterations_final = fin$param_final$num_iterations, semillas = PARAM$semillerio$cantidad)
}

reintentar_pendientes()
resumen <- rbindlist(resumen)
fwrite(resumen, "resumen_variantes.csv")
message(format(Sys.time(), "%a %b %d %X %Y"), "  terminado en ",
        round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
resumen

## Puntajes de Kaggle y elección del corte
Muestra la curva Public del WF7191 al lado del WF7190 (`PARAM$comparar_con`) y sugiere el corte estable (mejor promedio de 3 cortes vecinos) entre todas las variantes.

In [ ]:
descargar_envios <- TRUE   # FALSE para reusar el envios_kaggle.csv ya bajado

if (descargar_envios) {
  preparar_kaggle()
  # la CLI lista hasta 200 envios: bajo la lista completa con la API de Python, pagina por pagina
  writeLines(c(
    "import csv, sys",
    "from kaggle.api.kaggle_api_extended import KaggleApi",
    "from kagglesdk.competitions.types.competition_api_service import ApiListSubmissionsRequest",
    "from kagglesdk.competitions.types.competition_enums import SubmissionGroup, SubmissionSortBy",
    "api = KaggleApi()",
    "api.authenticate()",
    "envios, token = [], ''",
    "for pagina in range(50):",
    "    with api.build_kaggle_client() as kaggle:",
    "        req = ApiListSubmissionsRequest()",
    "        req.competition_name = sys.argv[1]",
    "        req.group = SubmissionGroup.SUBMISSION_GROUP_ALL",
    "        req.sort_by = SubmissionSortBy.SUBMISSION_SORT_BY_DATE",
    "        api._set_paging(req, 200, token, -1)",
    "        resp = kaggle.competitions.competition_api_client.list_submissions(req)",
    "    envios.extend(resp.submissions or [])",
    "    token = getattr(resp, 'next_page_token', '') or ''",
    "    if not token or not resp.submissions:",
    "        break",
    "with open(sys.argv[2], 'w', newline='', encoding='utf-8') as f:",
    "    w = csv.writer(f)",
    "    w.writerow(['ref', 'archivo', 'descripcion', 'fecha', 'estado', 'public'])",
    "    for s in envios:",
    "        w.writerow([s.ref, s.file_name, s.description, s.date, s.status, s.public_score])",
    "print(len(envios), 'envios en la competencia')"
  ), "bajar_envios_kaggle.py")
  invisible(system(paste("python3 bajar_envios_kaggle.py", PARAM$kaggle$competencia, "envios_kaggle.csv")))
}

# envios de este notebook y de los experimentos a comparar: KA<experimento>_<variante>_<envios>.csv
patron <- paste0("^KA(", paste(c(PARAM$experimento, PARAM$comparar_con), collapse = "|"), ")_(.+)_([0-9]+)\\.csv$")
envios_kaggle <- fread("envios_kaggle.csv", colClasses = "character")
envios_kaggle[, public := suppressWarnings(as.numeric(public))]
puntajes <- envios_kaggle[grepl(patron, archivo) & grepl("COMPLETE", estado) & !is.na(public)]
puntajes[, variante := paste0(sub(patron, "\\1", archivo), "_", sub(patron, "\\2", archivo))]
puntajes[, envios := as.integer(sub(patron, "\\3", archivo))]
setorder(puntajes, fecha)
puntajes <- puntajes[, .SD[.N], by = .(variante, envios)]   # si se subio dos veces, el mas reciente
fwrite(puntajes[, .(variante, envios, public, fecha, ref, archivo)], "puntajes_kaggle.csv")

# curva de ganancia Public por corte
curva <- dcast(puntajes, envios ~ variante, value.var = "public")
print(curva)

# el pico aislado del Public es ruido: elijo el corte con mejor promedio en una ventana de 3 cortes vecinos
setorder(puntajes, variante, envios)
puntajes[, ventana3 := frollmean(public, 3, align = "center"), by = variante]
mejor <- puntajes[!is.na(ventana3)][order(-ventana3)][1]
fwrite(puntajes, "puntajes_ventanas.csv")
message("Maximo Public: ", puntajes[which.max(public), paste0(variante, " en ", envios, " envios (", public, ")")])
message("Corte estable sugerido: ", mejor$variante, " en ", mejor$envios, " envios",
        "  (promedio de la ventana ", round(mejor$ventana3, 3), ", Public del corte ", mejor$public, ")")
message("Archivo a marcar como entrega final en Kaggle: ", mejor$archivo)